In [1]:
from pathlib import Path
from copy import deepcopy

import matplotlib.pyplot as plt
import matplotlib as mpl

from contact_plots import get_data_from_files, create_compare_plot

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder, convert_2_minutes=True)

In [3]:
cut_off_distances = ['1.0', '1.5', '2.0']

data_names_base = (
    'customer_customer_weight_over_contacts',
    'customer_customer_weight_over_agents',
    'customer_customer_contacts_over_agents',
    'staff_customer_weight_over_contacts',
    'staff_customer_weight_over_agents',
    'staff_customer_contacts_over_agents',
    'customer_staff_weight_over_agents',
    'customer_staff_contacts_over_agents',
)

parts_2_data_name = {}
for cut_off_distance in cut_off_distances:
    data_name_dict = {}
    for data_name in data_names_base:
        parts = data_name.split('_')
        interaction_type = '_'.join(parts[:2])
        data_type = '_'.join(parts[2:])
        if interaction_type not in data_name_dict:
            data_name_dict[interaction_type] = {}
            
        data_name_dict[interaction_type][data_type] = f'{cut_off_distance}_{data_name}'
    
    parts_2_data_name[cut_off_distance] = data_name_dict
    
measures_and_labels = (
    ('nn_nts_bvd', 'Base'),
    ('n_nts_bvd', 'Neighbourhoods'),
    ('nn_ts_bvd', 'Time slots'),
    ('nn_nts_svd', 'Shorter visits'),
    ('nn_nts_lvd', 'Longer visits')
)

base_measure_name = 'nn_nts_bvd'

infra_names = [
    'base_case',
    'busy_corner',
    'long_restaurant']



In [4]:
divider_config = {
    'left':0.75,  
    'gap_hor':1.05,
    'right':0.1,
    'bottom':1.15,
    'gap_ver':0.40,
    'v_legend':0.30,
    'top':0.10,
}

measure_names = [measure_name for measure_name, _ in measures_and_labels]
labels = [label for _, label in measures_and_labels]

flier_props = {'markersize':4, 'marker':'x', 'markeredgecolor':'0.8',
                  'linewidth':1, 'markeredgewidth':1}

box_plot_config = {'vert':True, 'notch':True, 'patch_artist':True, 'widths':0.8, 'flierprops':flier_props,
                   'whiskerprops':{'linewidth':2},'capprops':{'linewidth':2},
                    'boxprops':{'linewidth':2, 'fill':False}, 'medianprops':{'linewidth':2, 'color':'tab:orange'}}#, 'facecolor':'tab:blue'}}

violin_plot_config = {'widths':0.8, 'showmeans':False, 'showextrema':False, 'showmedians':False}
violin_body_plot_config = {'face_color':'tab:blue', 'alpha':0.7}

bar_plot_config = {'width':0.8, 'edgecolor':'k', 'linewidth':1}

bar_plot_legend_config = {'loc':'best', 'ncol':3, 'title':'Contact counts'}
bar_plot_text_config = {'fontsize':8, 'horizontalalignment':'center', 'verticalalignment':'center'}

main_ax_config = {'grid':{'visible':True, 'axis':'y'}, 'y_label_config':{'fontsize':10},
                 'x_tick_labels':labels, 'x_tick_labels_config':{'fontsize':10, 'rotation':45, 'ha':'right'}, 
                  'title_config':{'fontsize':12}}

legend_config = {'loc':'upper center', 'ncol':9}

line_plot_config = {}

v_line_config = {'linewidth':1.5, 'color':'0.6', 'linestyle':'--', 'zorder':2}

base_plot_config = {'main_type':'box_plot', 'measure_names':measure_names,
                   'main_ax_config':main_ax_config, 'v_line_config':v_line_config,
                    'legend_config':legend_config,
                    'bar_plot_config':bar_plot_config, 'bar_plot_legend_config':bar_plot_legend_config,
                   'bar_plot_text_config':bar_plot_text_config,
                   'violin_plot_config':violin_plot_config, 'violin_body_plot_config':violin_body_plot_config}

plot_types_to_y_labels_and_titles = (
    ('weight_over_contacts', 'Weight [min]', 'Weight over contacts'),
    ('weight_over_agents', 'Weight [min]', 'Weight over agents'),
    ('contacts_over_agents', 'Count [-]', 'Contacts over agent')
)

interaction_type_labels = {'customer_customer':'Customer - Customer', 
                           'customer_staff':'Customer - Staff', 
                           'staff_customer':'Staff - Customer'}

blues = mpl.colormaps['Blues']
reds = mpl.colormaps['Reds']
greens = mpl.colormaps['Greens']
bottom_border = 100
top_border = 30
border_sum = bottom_border + top_border

sub_group_2_color  = {
    'base_case':{
        '1.0':blues(int(bottom_border + (blues.N - border_sum)/2*0)), 
        '1.5':blues(int(bottom_border + (blues.N - border_sum)/2*1)),
        '2.0':blues(int(bottom_border + (blues.N - border_sum)/2*2))},
    'busy_corner':{
        '1.0':reds(int(bottom_border + (reds.N - border_sum)/2*0)), 
        '1.5':reds(int(bottom_border + (reds.N - border_sum)/2*1)),
        '2.0':reds(int(bottom_border + (reds.N - border_sum)/2*2))},
    'long_restaurant':{
        '1.0':greens(int(bottom_border + (greens.N - border_sum)/2*0)), 
        '1.5':greens(int(bottom_border + (greens.N - border_sum)/2*1)),
        '2.0':greens(int(bottom_border + (greens.N - border_sum)/2*2))},
}

fig_config_base = {}


In [5]:
%matplotlib wx

plt.style.use('dark_background')
#plt.style.use('ggplot')

data_type_all = 'weight_over_agents'
y_label_all =  'Weight [min]'

fig_config = deepcopy(fig_config_base)
fig_config.update({'title': f'All - {data_type_all}'})

plot_config = deepcopy(base_plot_config)
plots_config = []

for ii, interaction_type in enumerate(('customer_customer', 'staff_customer')):
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_ax_config'].update({'y_label':y_label_all, 'title':f'({chr(97+ii)}) {interaction_type_labels[interaction_type]}'})
    sub_groups = []
    for infra_name in infra_names:
        for cut_off_distance in cut_off_distances:
            data_name = parts_2_data_name[cut_off_distance][interaction_type][data_type]
            box_plot_config_sub = deepcopy(box_plot_config)
            box_plot_config_sub['boxprops'].update({'facecolor':sub_group_2_color[infra_name][cut_off_distance]})
            sub_group = {'infra_name':infra_name, 'data_name':data_name, 
                         'label':f'{infra_name} - {cut_off_distance}',
                        'box_plot_config':box_plot_config_sub}
            sub_groups.append(sub_group)
    plot_config_sub['sub_groups'] = sub_groups
            
    plots_config.append(plot_config_sub)
    
create_compare_plot(fig_config, divider_config, plots_config, infra_and_measure_name_2_data)

In [6]:
data_type_all = 'weight_over_contacts'
y_label_all =  'Weight [min]'

fig_config = deepcopy(fig_config_base)
fig_config.update({'title': f'All - {data_type_all}'})

plot_config = deepcopy(base_plot_config)
plots_config = []

for ii, interaction_type in enumerate(interaction_type_labels):
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_ax_config'].update({'y_label':y_label_all, 'title':f'({chr(97+ii)}) {interaction_type_labels[interaction_type]}'})
    sub_groups = []
    for infra_name in infra_names:
        for cut_off_distance in cut_off_distances:
            data_name = parts_2_data_name[cut_off_distance][interaction_type][data_type]
            box_plot_config_sub = deepcopy(box_plot_config)
            box_plot_config_sub['boxprops'].update({'facecolor':sub_group_2_color[infra_name][cut_off_distance]})
            sub_group = {'infra_name':infra_name, 'data_name':data_name, 
                         'label':f'{infra_name} - {cut_off_distance}',
                        'box_plot_config':box_plot_config_sub}
            sub_groups.append(sub_group)
    plot_config_sub['sub_groups'] = sub_groups
            
    plots_config.append(plot_config_sub)
    
create_compare_plot(fig_config, divider_config, plots_config, infra_and_measure_name_2_data)

In [10]:
data_type_all = 'contacts_over_agents'
y_label_all =  'Count [-]'

fig_config = deepcopy(fig_config_base)
fig_config.update({'title': f'All - {data_type_all}'})

plot_config = deepcopy(base_plot_config)
plots_config = []

for ii, interaction_type in enumerate(interaction_type_labels):
    plot_config_sub = deepcopy(plot_config)
    plot_config_sub['main_ax_config'].update({'y_label':y_label_all, 'title':f'({chr(97+ii)}) {interaction_type_labels[interaction_type]}'})
    sub_groups = []
    for infra_name in infra_names:
        for cut_off_distance in cut_off_distances:            
            data_name = parts_2_data_name[cut_off_distance][interaction_type][data_type]
            box_plot_config_sub = deepcopy(box_plot_config)
            box_plot_config_sub['boxprops'].update({'facecolor':sub_group_2_color[infra_name][cut_off_distance]})
            sub_group = {'infra_name':infra_name, 'data_name':data_name, 
                         'label':f'{infra_name} - {cut_off_distance}',
                        'box_plot_config':box_plot_config_sub}
            if interaction_type == 'customer_staff':
                line_plot_config_sub = deepcopy(line_plot_config)
                line_plot_config_sub.update({'color':sub_group_2_color[infra_name][cut_off_distance]})
                sub_group.update({'line_plot_config':line_plot_config_sub})            
            sub_groups.append(sub_group)
            
    if interaction_type == 'customer_staff':
        plot_config_sub['main_type'] = 'bar_border_line'
        plot_config_sub['main_ax_config']['y_label'] = 'Distribution of counts [-]'
        
    plot_config_sub['sub_groups'] = sub_groups
    plots_config.append(plot_config_sub)
    
print(len(plots_config))
create_compare_plot(fig_config, divider_config, plots_config, infra_and_measure_name_2_data)


3
base_case - 1.0
[1, 2, 3]
base_case - 1.0 - (1 - 2)
[0, 0, 0, 0, 0]
[0.22611876075731496, 0.5685250369024639, 0.23911187019641333, 0.2342710587480965, 0.18027817403708987]
[0.22611876075731496, 0.5685250369024639, 0.23911187019641333, 0.2342710587480965, 0.18027817403708987]
base_case - 1.0 - (2 - 3)
[0.22611876075731496, 0.5685250369024639, 0.23911187019641333, 0.2342710587480965, 0.18027817403708987]
[0.38543459552495696, 0.8874758714658794, 0.4036080273270709, 0.39392482167187626, 0.33915834522111266]
[0.159315834767642, 0.31895083456341544, 0.16449615713065754, 0.15965376292377975, 0.1588801711840228]
busy_corner - 1.0
[1, 2, 3]
busy_corner - 1.0 - (1 - 2)
[0, 0, 0, 0, 0]
[0.15595728737354814, 0.47694282610842303, 0.19664245042214804, 0.1719281534902708, 0.12664826740263724]
[0.15595728737354814, 0.47694282610842303, 0.19664245042214804, 0.1719281534902708, 0.12664826740263724]
busy_corner - 1.0 - (2 - 3)
[0.15595728737354814, 0.47694282610842303, 0.19664245042214804, 0.171928153

In [ ]:
plt.show()